# 本地RPS输入复核
输入只读，生产数据不修复。完整SQL/Python见相邻audit.py和cache_check.py。先运行两个脚本生成隔离诊断输出。

In [ ]:
from pathlib import Path
import json
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'rps_analyzer').is_dir())
out=root/'outputs/diagnostics/rps_validity_20261024'
audit=json.loads((out/'audit.json').read_text(encoding='utf-8'))
cache=json.loads((out/'cache_check.json').read_text(encoding='utf-8'))
assert all(v['max_abs_delta']==0 for v in cache['rps_formula'].values())
print(audit['tdx']['issues'],cache['rps_formula'])

In [ ]:
import duckdb
with duckdb.connect(str(root/'data/hithink/market.duckdb'),read_only=True) as con:
    raw=con.execute("SELECT date,close,volume,amount FROM v_daily WHERE thscode='000001.SZ' AND date='2025-11-27'").fetchdf()
display(raw)
# Same common stock universe; raw-to-qfq top-decile changes, not portfolio performance.
for n in ('120','250'):
    r=audit['current_rps'][n]
    print(n,r['common_universe'],r['qfq_top10_size']-r['raw_vs_qfq_top10_overlap'])